# 04 — Visualize embeddings and prepare the demo story

This notebook creates actual-data visuals from the multimodal artifact.

It is the notebook to use for your recorded demo visuals.

In [ ]:
import os
REPO_DIR = "/content/nifty50-multimodal-transformer"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

OUTPUT_DIR = Path("data/processed/real_world_demo")
data = np.load(OUTPUT_DIR / "real_world_multimodal_samples.npz", allow_pickle=False)

for key in data.files:
    print(f"{key:16s}", data[key].shape, data[key].dtype)

In [ ]:
# Build one vector per sample for each modality.
tabular_vec = data["tabular_tokens"].mean(axis=1)
image_vec = data["image_tokens"]
text_vec = data["text_tokens"]
kg_vec = data["kg_tokens"]

modalities = {
    "tabular": tabular_vec,
    "image": image_vec,
    "text": text_vec,
    "kg": kg_vec,
}

In [ ]:
# PCA projection per modality. These are actual vectors from the generated artifact.
rows = []
for modality, values in modalities.items():
    n_components = 2 if values.shape[1] >= 2 else 1
    projection = PCA(n_components=n_components).fit_transform(values)
    if n_components == 1:
        projection = np.c_[projection[:, 0], np.zeros(values.shape[0])]
    for i, (x, y) in enumerate(projection):
        rows.append({
            "sample_id": i,
            "stock_id": str(data["stock_ids"][i]),
            "end_date": str(data["end_dates"][i]),
            "label": int(data["y"][i]),
            "modality": modality,
            "x": float(x),
            "y": float(y),
        })

projection_df = pd.DataFrame(rows)
projection_csv = OUTPUT_DIR / "modality_embedding_projection.csv"
projection_df.to_csv(projection_csv, index=False)
display(projection_df.head())
print("Wrote:", projection_csv)

In [ ]:
# Plot actual embedding projections.
fig, ax = plt.subplots(figsize=(10, 7))
for modality, frame in projection_df.groupby("modality"):
    ax.scatter(frame["x"], frame["y"], label=modality, alpha=0.75)

ax.set_title("Actual modality embedding projections from real-world demo artifact")
ax.set_xlabel("PCA dimension 1")
ax.set_ylabel("PCA dimension 2")
ax.legend()
plt.tight_layout()

output_path = OUTPUT_DIR / "modality_embedding_projection.png"
plt.savefig(output_path, dpi=160)
print("Wrote:", output_path)
plt.show()

In [ ]:
# Build a small sample gallery markdown linking real charts to labels.
chart_dir = OUTPUT_DIR / "charts"
gallery_path = OUTPUT_DIR / "sample_gallery.md"

lines = ["# Real Demo Sample Gallery", ""]
for i in range(min(10, len(data["stock_ids"]))):
    stock_id = str(data["stock_ids"][i])
    end_date = pd.Timestamp(data["end_dates"][i]).strftime("%Y%m%d")
    label = int(data["y"][i])
    chart_name = f"{stock_id}_{end_date}.png"
    lines.extend([
        f"## Sample {i}: {stock_id} / {end_date}",
        "",
        f"- label: `{label}`",
        f"- chart: `charts/{chart_name}`",
        "",
        f"![{chart_name}](charts/{chart_name})",
        "",
    ])

gallery_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote:", gallery_path)

In [ ]:
# Demo summary markdown.
summary_path = OUTPUT_DIR / "demo_visual_summary.md"
lines = [
    "# Demo Visual Summary",
    "",
    "Generated from actual real-world demo artifacts.",
    "",
    "## Files",
    "",
    "- `modality_embedding_projection.csv`",
    "- `modality_embedding_projection.png`",
    "- `ablation_metrics_bar_chart.png` if notebook 03 was run",
    "- `sample_gallery.md`",
    "",
    "## What to say in the recording",
    "",
    "1. Raw data becomes aligned tensors.",
    "2. Each modality creates a vector representation.",
    "3. Fusion combines those vectors in a shared embedding space.",
    "4. Ablations compare modality combinations.",
    "5. Backtesting is the next evidence layer and should not be claimed yet.",
]
summary_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote:", summary_path)

## Checkpoint output

This notebook writes actual-data demo visuals:

```text
data/processed/real_world_demo/modality_embedding_projection.csv
data/processed/real_world_demo/modality_embedding_projection.png
data/processed/real_world_demo/sample_gallery.md
data/processed/real_world_demo/demo_visual_summary.md
```

These are safe to use in the recording because they come from the real demo artifact, not generated conceptual slides.